# Predicting Student Test Scores 
## Score: 8.56302

In [1]:
import os
import hashlib
import numpy as np
import pandas as pd
import xgboost as xgb
import warnings

from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)

In [2]:
TRAIN_PATH = 'playground-series-s6e1/train.csv'
TEST_PATH = 'playground-series-s6e1/test.csv'
ORIGINAL_PATH = 'Exam_Score_Prediction.csv'

TARGET = 'exam_score'
ID_COL = 'id'

N_FOLDS = 8
RANDOM_STATE = 80085
XGB_SEEDS = [42]  # single-seed final model to reduce runtime

if not os.path.exists(ORIGINAL_PATH):
    raise FileNotFoundError(f'Original dataset not found at {ORIGINAL_PATH}')

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
original_df = pd.read_csv(ORIGINAL_PATH)

print(f'Train:    {train_df.shape}')
print(f'Test:     {test_df.shape}')
print(f'Original: {original_df.shape}')
print(f'XGB Seeds: {XGB_SEEDS}')

base_features = [c for c in train_df.columns if c not in [TARGET, ID_COL]]
cat_features = train_df.select_dtypes('object').columns.tolist()

print(f'\nBase features: {len(base_features)}')
print(f'Categorical:   {cat_features}')

Train:    (630000, 13)
Test:     (270000, 12)
Original: (20000, 13)
XGB Seeds: [42]

Base features: 11
Categorical:   ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']


In [3]:
def engineer_features(df, base_cols):
    out = df.copy()
    eps = 1e-5
    
    study = out['study_hours'].clip(lower=0)
    attend = out['class_attendance'].clip(lower=0)
    sleep = out['sleep_hours'].clip(lower=0)
    
    out['study_hours_squared'] = out['study_hours'] ** 2
    out['class_attendance_squared'] = out['class_attendance'] ** 2
    out['sleep_hours_squared'] = out['sleep_hours'] ** 2
    out['age_squared'] = out['age'] ** 2
    
    out['log_study_hours'] = np.log1p(study)
    out['log_class_attendance'] = np.log1p(attend)
    out['log_sleep_hours'] = np.log1p(sleep)
    
    out['sqrt_study_hours'] = np.sqrt(study)
    out['sqrt_class_attendance'] = np.sqrt(attend)
    
    out['study_hours_times_attendance'] = out['study_hours'] * out['class_attendance']
    out['study_hours_times_sleep'] = out['study_hours'] * out['sleep_hours']
    out['attendance_times_sleep'] = out['class_attendance'] * out['sleep_hours']
    out['age_times_study_hours'] = out['age'] * out['study_hours']
    
    out['study_hours_over_sleep'] = out['study_hours'] / (out['sleep_hours'] + eps)
    out['attendance_over_sleep'] = out['class_attendance'] / (out['sleep_hours'] + eps)
    out['attendance_over_study'] = out['class_attendance'] / (out['study_hours'] + eps)
    
    ordinal_maps = {
        'sleep_quality': {'poor': 0, 'average': 1, 'good': 2},
        'facility_rating': {'low': 0, 'medium': 1, 'high': 2},
        'exam_difficulty': {'easy': 0, 'moderate': 1, 'hard': 2}
    }
    for col, mapping in ordinal_maps.items():
        out[f'{col}_numeric'] = out[col].map(mapping).fillna(1).astype(int)
    
    out['study_hours_times_sleep_quality'] = out['study_hours'] * out['sleep_quality_numeric']
    out['attendance_times_facility'] = out['class_attendance'] * out['facility_rating_numeric']
    out['sleep_hours_times_difficulty'] = out['sleep_hours'] * out['exam_difficulty_numeric']
    out['facility_x_sleepq'] = out['facility_rating_numeric'] * out['sleep_quality_numeric']
    out['difficulty_x_facility'] = out['exam_difficulty_numeric'] * out['facility_rating_numeric']
    
    out['high_att_high_study'] = ((out['class_attendance'] >= 90) & (out['study_hours'] >= 6)).astype(int)
    out['ideal_sleep_flag'] = ((out['sleep_hours'] >= 7) & (out['sleep_hours'] <= 9)).astype(int)
    out['high_study_flag'] = (out['study_hours'] >= 7).astype(int)
    
    out['efficiency'] = (out['study_hours'] * out['class_attendance']) / (out['sleep_hours'] + 1)
    out['efficiency_log'] = np.log1p(out['efficiency'])
    
    out['sleep_gap_8'] = (out['sleep_hours'] - 8.0).abs()
    out['attendance_gap_100'] = (out['class_attendance'] - 100.0).abs()
    out['study_gap_7'] = (out['study_hours'] - 7.0).abs()
    
    out['study_hours_times_exam_difficulty'] = out['study_hours'] * out['exam_difficulty_numeric']
    out['undersleep_flag'] = (out['sleep_hours'] < 6).astype(int)
    out['oversleep_flag'] = (out['sleep_hours'] > 9).astype(int)
    out['attendance_low_flag'] = (out['class_attendance'] < 80).astype(int)
    
    out['study_bin_num'] = pd.cut(out['study_hours'], bins=5, labels=False).astype(int)
    out['attendance_bin_num'] = pd.cut(out['class_attendance'], bins=5, labels=False).astype(int)
    out['sleep_bin_num'] = pd.cut(out['sleep_hours'], bins=5, labels=False).astype(int)
    out['age_bin_num'] = pd.cut(out['age'], bins=5, labels=False).astype(int)
    try:
        out['study_quintile'] = pd.qcut(out['study_hours'], q=5, labels=False, duplicates='drop').astype(float).fillna(0).astype(int)
        out['attendance_quintile'] = pd.qcut(out['class_attendance'], q=5, labels=False, duplicates='drop').astype(float).fillna(0).astype(int)
    except Exception:
        out['study_quintile'] = out['study_bin_num']
        out['attendance_quintile'] = out['attendance_bin_num']
    
    engineered_cols = [
        'study_hours_squared', 'class_attendance_squared', 'sleep_hours_squared', 'age_squared',
        'log_study_hours', 'log_class_attendance', 'log_sleep_hours',
        'sqrt_study_hours', 'sqrt_class_attendance',
        'study_hours_times_attendance', 'study_hours_times_sleep', 'attendance_times_sleep',
        'age_times_study_hours',
        'study_hours_over_sleep', 'attendance_over_sleep', 'attendance_over_study',
        'sleep_quality_numeric', 'facility_rating_numeric', 'exam_difficulty_numeric',
        'study_hours_times_sleep_quality', 'attendance_times_facility', 'sleep_hours_times_difficulty',
        'facility_x_sleepq', 'difficulty_x_facility',
        'high_att_high_study', 'ideal_sleep_flag', 'high_study_flag',
        'efficiency', 'efficiency_log',
        'sleep_gap_8', 'attendance_gap_100', 'study_gap_7',
        'study_hours_times_exam_difficulty', 'undersleep_flag', 'oversleep_flag', 'attendance_low_flag',
        'study_bin_num', 'attendance_bin_num', 'sleep_bin_num', 'age_bin_num',
        'study_quintile', 'attendance_quintile'
    ]
    
    return out[base_cols + engineered_cols], engineered_cols

X_train, engineered_cols = engineer_features(train_df, base_features)
X_test, _ = engineer_features(test_df, base_features)
X_orig, _ = engineer_features(original_df, base_features)

y_train = train_df[TARGET].reset_index(drop=True)
y_orig = original_df[TARGET].reset_index(drop=True)

full_data = pd.concat([X_train, X_test, X_orig], axis=0, ignore_index=True)
for col in engineered_cols:
    full_data[col] = full_data[col].astype(float)

n_train, n_test = len(train_df), len(test_df)
X = full_data.iloc[:n_train].copy()
X_test = full_data.iloc[n_train:n_train + n_test].copy()
X_original = full_data.iloc[n_train + n_test:].copy()

print(f'Engineered features: {len(engineered_cols)}')
print(f'Total features:      {X.shape[1]} (11 base + {len(engineered_cols)} engineered)')

Engineered features: 42
Total features:      53 (11 base + 42 engineered)


In [4]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

oof_ridge = np.zeros(len(X))
test_preds_ridge = np.zeros((len(X_test), N_FOLDS))
orig_preds_ridge = np.zeros(len(X_original))

ridge_alphas = np.logspace(-3, 3, 20)

print('Training Ridge Regression')
print('-' * 40)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_train), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    X_tr_aug = pd.concat([X_tr, X_original], axis=0)
    y_tr_aug = pd.concat([y_tr, y_orig], axis=0)
    
    encoder = TargetEncoder(smooth='auto', target_type='continuous')
    X_tr_enc = X_tr_aug.copy()
    X_val_enc = X_val.copy()
    X_test_enc = X_test.copy()
    
    X_tr_enc[cat_features] = encoder.fit_transform(X_tr_aug[cat_features], y_tr_aug)
    X_val_enc[cat_features] = encoder.transform(X_val[cat_features])
    X_test_enc[cat_features] = encoder.transform(X_test[cat_features])
    
    ridge = RidgeCV(alphas=ridge_alphas, cv=5, scoring='neg_root_mean_squared_error')
    ridge.fit(X_tr_enc, y_tr_aug.values.ravel())
    
    oof_ridge[val_idx] = np.clip(ridge.predict(X_val_enc), 0, 100)
    test_preds_ridge[:, fold - 1] = np.clip(ridge.predict(X_test_enc), 0, 100)
    orig_preds_ridge += np.clip(ridge.predict(X_tr_enc.iloc[-len(X_original):]), 0, 100) / N_FOLDS
    
    rmse = np.sqrt(mean_squared_error(y_val, oof_ridge[val_idx]))
    print(f'Fold {fold:2d} | RMSE: {rmse:.6f}')

ridge_oof_rmse = np.sqrt(mean_squared_error(y_train, oof_ridge))
print(f'\nRidge OOF RMSE: {ridge_oof_rmse:.6f}')

Training Ridge Regression
----------------------------------------
Fold  1 | RMSE: 8.875499
Fold  2 | RMSE: 8.848584
Fold  3 | RMSE: 8.922794
Fold  4 | RMSE: 8.902121
Fold  5 | RMSE: 8.881280
Fold  6 | RMSE: 8.915585
Fold  7 | RMSE: 8.899282
Fold  8 | RMSE: 8.890024

Ridge OOF RMSE: 8.891924


In [5]:
for col in base_features:
    full_data[col] = full_data[col].astype(str).astype('category')
for col in engineered_cols:
    full_data[col] = full_data[col].astype(float)

X_xgb = full_data.iloc[:n_train].copy()
X_test_xgb = full_data.iloc[n_train:n_train + n_test].copy()
X_orig_xgb = full_data.iloc[n_train + n_test:].copy()

X_xgb['ridge_pred'] = oof_ridge
X_test_xgb['ridge_pred'] = test_preds_ridge.mean(axis=1)
X_orig_xgb['ridge_pred'] = orig_preds_ridge

print(f'Feature count before pruning: {X_xgb.shape[1]}')

Feature count before pruning: 54


In [6]:
# Quick feature importance check (single fold)
print('Computing feature importance (single fold)...')

quick_params = {
    'n_estimators': 2000,
    'learning_rate': 0.01,
    'max_depth': 7,
    'subsample': 0.8,
    'colsample_bytree': 0.6,
    'tree_method': 'hist',
    'enable_categorical': True,
    'eval_metric': 'rmse',
    'early_stopping_rounds': 50,
    'random_state': 42
}

train_idx, val_idx = next(iter(kf.split(X_xgb, y_train)))
X_tr_q, X_val_q = X_xgb.iloc[train_idx], X_xgb.iloc[val_idx]
y_tr_q, y_val_q = y_train.iloc[train_idx], y_train.iloc[val_idx]

X_tr_aug_q = pd.concat([X_tr_q, X_orig_xgb], axis=0)
y_tr_aug_q = pd.concat([y_tr_q, y_orig], axis=0)

quick_model = xgb.XGBRegressor(**quick_params)
quick_model.fit(X_tr_aug_q, y_tr_aug_q, eval_set=[(X_val_q, y_val_q)], verbose=False)

importance = quick_model.feature_importances_
feature_names = X_xgb.columns.tolist()
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importance}).sort_values('importance', ascending=False)

# Keep top 98% of features by cumulative importance
imp_df['cum_importance'] = imp_df['importance'].cumsum() / imp_df['importance'].sum()
keep_features = imp_df[imp_df['cum_importance'] <= 0.98]['feature'].tolist()

# Always keep ridge_pred and base categorical features
must_keep = ['ridge_pred'] + base_features
for f in must_keep:
    if f not in keep_features:
        keep_features.append(f)

dropped = [f for f in feature_names if f not in keep_features]
print(f'\nDropping {len(dropped)} low-importance features: {dropped}')
print(f'Keeping {len(keep_features)} features')

X_xgb = X_xgb[keep_features]
X_test_xgb = X_test_xgb[keep_features]
X_orig_xgb = X_orig_xgb[keep_features]

print(f'\nFeature count after pruning: {X_xgb.shape[1]}')

Computing feature importance (single fold)...

Dropping 23 low-importance features: ['sleep_hours_squared', 'age_squared', 'log_sleep_hours', 'sqrt_study_hours', 'age_times_study_hours', 'study_hours_over_sleep', 'attendance_over_sleep', 'attendance_over_study', 'facility_rating_numeric', 'exam_difficulty_numeric', 'sleep_hours_times_difficulty', 'difficulty_x_facility', 'high_att_high_study', 'ideal_sleep_flag', 'high_study_flag', 'sleep_gap_8', 'study_hours_times_exam_difficulty', 'undersleep_flag', 'oversleep_flag', 'study_bin_num', 'attendance_bin_num', 'sleep_bin_num', 'age_bin_num']
Keeping 31 features

Feature count after pruning: 31


In [7]:
# Initial XGBoost training (single seed for pseudo-label generation)
xgb_params_base = {
    'n_estimators': 25000,
    'learning_rate': 0.003,
    'max_depth': 9,
    'subsample': 0.78,
    'colsample_bytree': 0.55,
    'colsample_bynode': 0.65,
    'reg_lambda': 6,
    'reg_alpha': 0.15,
    'min_child_weight': 6,
    'tree_method': 'hist',
    'enable_categorical': True,
    'eval_metric': 'rmse',
    'early_stopping_rounds': 200,
    'random_state': 42
}

test_preds_xgb = []
oof_xgb = np.zeros(len(X_xgb))

print('Training XGBoost (initial, for pseudo-labels)')
print('-' * 40)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_xgb, y_train), 1):
    print(f'\nFold {fold}/{N_FOLDS}')
    
    X_tr, X_val = X_xgb.iloc[train_idx], X_xgb.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    X_tr_aug = pd.concat([X_tr, X_orig_xgb], axis=0)
    y_tr_aug = pd.concat([y_tr, y_orig], axis=0)
    
    model = xgb.XGBRegressor(**xgb_params_base)
    model.fit(
        X_tr_aug, y_tr_aug,
        eval_set=[(X_val, y_val)],
        verbose=1000
    )
    
    oof_xgb[val_idx] = model.predict(X_val)
    test_preds_xgb.append(model.predict(X_test_xgb))
    
    rmse = np.sqrt(mean_squared_error(y_val, oof_xgb[val_idx]))
    print(f'Validation RMSE: {rmse:.5f}')

xgb_oof_rmse = np.sqrt(mean_squared_error(y_train, oof_xgb))
print(f'\nXGBoost OOF RMSE: {xgb_oof_rmse:.5f}')

Training XGBoost (initial, for pseudo-labels)
----------------------------------------

Fold 1/8
[0]	validation_0-rmse:18.91324
[1000]	validation_0-rmse:8.67786
[2000]	validation_0-rmse:8.58372
[2819]	validation_0-rmse:8.58157
Validation RMSE: 8.58148

Fold 2/8
[0]	validation_0-rmse:18.82848
[1000]	validation_0-rmse:8.65625
[2000]	validation_0-rmse:8.56665
[3000]	validation_0-rmse:8.56344
[3165]	validation_0-rmse:8.56372
Validation RMSE: 8.56331

Fold 3/8
[0]	validation_0-rmse:18.88006
[1000]	validation_0-rmse:8.73468
[2000]	validation_0-rmse:8.64631
[3000]	validation_0-rmse:8.64345
[3269]	validation_0-rmse:8.64354
Validation RMSE: 8.64335

Fold 4/8
[0]	validation_0-rmse:18.87572
[1000]	validation_0-rmse:8.70991
[2000]	validation_0-rmse:8.62742
[2956]	validation_0-rmse:8.62584
Validation RMSE: 8.62559

Fold 5/8
[0]	validation_0-rmse:18.89600
[1000]	validation_0-rmse:8.69422
[2000]	validation_0-rmse:8.60758
[2703]	validation_0-rmse:8.60623
Validation RMSE: 8.60607

Fold 6/8
[0]	validati

In [8]:
# Pseudo-labeling with slightly wider confidence range
print('Pseudo-labeling...')

test_preds_avg = np.mean(test_preds_xgb, axis=0)

y_mean = y_train.mean()
y_std = y_train.std()

# Slightly wider range: 0.6 std (was 0.5)
PSEUDO_STD = 0.6
lower_bound = y_mean - PSEUDO_STD * y_std
upper_bound = y_mean + PSEUDO_STD * y_std

confident_mask = (test_preds_avg >= lower_bound) & (test_preds_avg <= upper_bound)
n_pseudo = confident_mask.sum()
print(f'Confident test samples: {n_pseudo} ({100*n_pseudo/len(test_preds_avg):.1f}%)')
print(f'Confidence range: [{lower_bound:.1f}, {upper_bound:.1f}]')

X_pseudo = X_test_xgb[confident_mask].copy()
y_pseudo = pd.Series(test_preds_avg[confident_mask])

print(f'Combined training size: {len(X_xgb)} + {len(X_orig_xgb)} + {len(X_pseudo)} = {len(X_xgb) + len(X_orig_xgb) + len(X_pseudo)}')

Pseudo-labeling...
Confident test samples: 118740 (44.0%)
Confidence range: [51.2, 73.9]
Combined training size: 630000 + 20000 + 118740 = 768740


In [9]:
# Final training with pseudo-labels + 2-seed averaging
print('\nFinal training with pseudo-labels + 2-seed averaging')
print('=' * 60)

all_oof_final = []
all_test_final = []

for seed_idx, seed in enumerate(XGB_SEEDS, 1):
    print(f'\n{"="*60}')
    print(f'SEED {seed} ({seed_idx}/{len(XGB_SEEDS)})')
    print(f'{"="*60}')
    
    xgb_params = {**xgb_params_base, 'random_state': seed}
    
    test_preds_seed = []
    oof_seed = np.zeros(len(X_xgb))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_xgb, y_train), 1):
        print(f'\nFold {fold}/{N_FOLDS}')
        
        X_tr, X_val = X_xgb.iloc[train_idx], X_xgb.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Augment with original + pseudo
        X_tr_aug = pd.concat([X_tr, X_orig_xgb, X_pseudo], axis=0)
        y_tr_aug = pd.concat([y_tr, y_orig, y_pseudo], axis=0)
        
        model = xgb.XGBRegressor(**xgb_params)
        model.fit(
            X_tr_aug, y_tr_aug,
            eval_set=[(X_val, y_val)],
            verbose=1000
        )
        
        oof_seed[val_idx] = model.predict(X_val)
        test_preds_seed.append(model.predict(X_test_xgb))
        
        rmse = np.sqrt(mean_squared_error(y_val, oof_seed[val_idx]))
        print(f'Validation RMSE: {rmse:.5f}')
    
    seed_oof_rmse = np.sqrt(mean_squared_error(y_train, oof_seed))
    print(f'\nSeed {seed} OOF RMSE: {seed_oof_rmse:.5f}')
    
    all_oof_final.append(oof_seed)
    all_test_final.append(np.mean(test_preds_seed, axis=0))

# Average across seeds
oof_final = np.mean(all_oof_final, axis=0)
test_final = np.mean(all_test_final, axis=0)

final_oof_rmse = np.sqrt(mean_squared_error(y_train, oof_final))
print(f'\n{"="*60}')
print(f'COMBINED OOF RMSE ({len(XGB_SEEDS)} seeds): {final_oof_rmse:.5f}')
print(f'{"="*60}')


Final training with pseudo-labels + 2-seed averaging

SEED 42 (1/1)

Fold 1/8
[0]	validation_0-rmse:18.91407
[1000]	validation_0-rmse:8.67307
[2000]	validation_0-rmse:8.56957
[3000]	validation_0-rmse:8.56423
[3369]	validation_0-rmse:8.56409
Validation RMSE: 8.56399

Fold 2/8
[0]	validation_0-rmse:18.82830
[1000]	validation_0-rmse:8.65152
[2000]	validation_0-rmse:8.55319
[3000]	validation_0-rmse:8.54678
[4000]	validation_0-rmse:8.54536
[4183]	validation_0-rmse:8.54549
Validation RMSE: 8.54530

Fold 3/8
[0]	validation_0-rmse:18.88041
[1000]	validation_0-rmse:8.73256
[2000]	validation_0-rmse:8.63418
[3000]	validation_0-rmse:8.62851
[3628]	validation_0-rmse:8.62796
Validation RMSE: 8.62785

Fold 4/8
[0]	validation_0-rmse:18.87637
[1000]	validation_0-rmse:8.70671
[2000]	validation_0-rmse:8.61339
[3000]	validation_0-rmse:8.60858
[3680]	validation_0-rmse:8.60822
Validation RMSE: 8.60795

Fold 5/8
[0]	validation_0-rmse:18.89589
[1000]	validation_0-rmse:8.68884
[2000]	validation_0-rmse:8.59295

In [10]:
# Post-processing: optimized Ridge blend + group bias correction
print('Post-processing...')

ridge_test_avg = test_preds_ridge.mean(axis=1)

# 1) Optimize small Ridge blend weight on OOF
base_oof = oof_final
base_test = test_final

best_w = 0.0
best_rmse = np.sqrt(mean_squared_error(y_train, base_oof))
print(f'Base XGB OOF RMSE (no Ridge blend): {best_rmse:.5f}')

for w in np.linspace(0.0, 0.06, 7):  # 0.00, 0.01, ..., 0.06
    blended_oof_try = (1 - w) * base_oof + w * oof_ridge
    rmse = np.sqrt(mean_squared_error(y_train, blended_oof_try))
    print(f'  Ridge weight {w:.3f} -> OOF RMSE {rmse:.5f}')
    if rmse < best_rmse - 1e-5:
        best_rmse = rmse
        best_w = w

print(f'Best Ridge weight: {best_w:.3f} (OOF RMSE {best_rmse:.5f})')

blended_oof = (1 - best_w) * base_oof + best_w * oof_ridge
blended_test = (1 - best_w) * base_test + best_w * ridge_test_avg

# 2) Group-wise residual bias correction (course x exam_difficulty)
def apply_group_residual_correction(oof, test_pred, train_df, test_df, target_col, group_cols, min_count=8000):
    df_tr = train_df.copy()
    df_tr['oof'] = oof
    df_tr['resid'] = df_tr[target_col] - df_tr['oof']

    grp = df_tr.groupby(group_cols)['resid'].agg(['mean', 'count']).reset_index()
    global_mean = df_tr['resid'].mean()

    # Frequency-based shrinkage toward global mean
    grp['weight'] = grp['count'] / (grp['count'] + min_count)
    grp['adj'] = global_mean + grp['weight'] * (grp['mean'] - global_mean)

    # Train adjustment
    tr_merge = train_df[group_cols].merge(grp[group_cols + ['adj']], on=group_cols, how='left')
    tr_adj = tr_merge['adj'].fillna(global_mean).values
    oof_corr = oof + tr_adj

    # Test adjustment
    te_merge = test_df[group_cols].merge(grp[group_cols + ['adj']], on=group_cols, how='left')
    te_adj = te_merge['adj'].fillna(global_mean).values
    test_corr = test_pred + te_adj

    return oof_corr, test_corr

group_cols = ['course', 'exam_difficulty']
oof_gc, test_gc = apply_group_residual_correction(
    blended_oof, blended_test,
    train_df, test_df,
    TARGET,
    group_cols=group_cols,
    min_count=8000,
)

rmse_gc = np.sqrt(mean_squared_error(y_train, oof_gc))
print(f'Group-corrected OOF RMSE ({" x ".join(group_cols)}): {rmse_gc:.5f}')

# Choose best of (blended vs group-corrected)
if rmse_gc < best_rmse:
    print('Using group bias correction')
    final_oof = oof_gc
    final_test = test_gc
    final_rmse = rmse_gc
else:
    print('Group correction did not help, using blended only')
    final_oof = blended_oof
    final_test = blended_test
    final_rmse = best_rmse

final_preds = final_test

print(f'\nFinal Model Performance')
print('-' * 40)
print(f'Ridge OOF RMSE:   {ridge_oof_rmse:.6f}')
print(f'XGB OOF RMSE:     {final_oof_rmse:.5f}')
print(f'Final OOF RMSE:   {final_rmse:.5f}')

Post-processing...
Base XGB OOF RMSE (no Ridge blend): 8.59237
  Ridge weight 0.000 -> OOF RMSE 8.59237
  Ridge weight 0.010 -> OOF RMSE 8.59241
  Ridge weight 0.020 -> OOF RMSE 8.59251
  Ridge weight 0.030 -> OOF RMSE 8.59268
  Ridge weight 0.040 -> OOF RMSE 8.59290
  Ridge weight 0.050 -> OOF RMSE 8.59319
  Ridge weight 0.060 -> OOF RMSE 8.59354
Best Ridge weight: 0.000 (OOF RMSE 8.59237)
Group-corrected OOF RMSE (course x exam_difficulty): 8.59221
Using group bias correction

Final Model Performance
----------------------------------------
Ridge OOF RMSE:   8.891924
XGB OOF RMSE:     8.59237
Final OOF RMSE:   8.59221


In [11]:
test_ids = test_df[ID_COL].values
final_preds = np.clip(final_preds, y_train.min(), y_train.max())

submission = pd.DataFrame({ID_COL: test_ids, TARGET: final_preds})
submission.to_csv('submission.csv', index=False)

with open('submission.csv', 'rb') as f:
    md5 = hashlib.md5(f.read()).hexdigest()

print('submission.csv')
print('md5', md5)
print(f'pred_mean {final_preds.mean():.4f}')
print(f'pred_std {final_preds.std():.4f}')
print(f'pred_min {final_preds.min():.4f}')
print(f'pred_max {final_preds.max():.4f}')

submission.csv
md5 2a8aff46b8fbd983fcec4847a25a59fd
pred_mean 62.5255
pred_std 16.8031
pred_min 19.5990
pred_max 100.0000
